<a href="https://colab.research.google.com/github/Swas26/NLP-ESG/blob/main/NLP_Assessment_3_Filtering_and_Scoring.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install transformers torch
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/NLP Assessment_3/combined.csv')
print(f"Loaded {len(df)} rows")
print(df['source_type'].value_counts())

In [ ]:
from transformers import pipeline

detector = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-detector'
)

def is_climate_related(text):
    text = str(text)[:1500]
    try:
        result = detector(text)[0]
        return result['label'] == 'yes'
    except:
        return False

print("Filtering with ClimateBERT detector...")
df['is_relevant'] = df['text'].apply(is_climate_related)

before = len(df)
filtered_df = df[df['is_relevant'] == True].drop(columns=['is_relevant'])
after = len(filtered_df)

print(f"Kept {after} / {before} rows ({round(after/before*100)}% relevant)")
print(filtered_df['source_type'].value_counts())

filtered_df.to_csv('/content/drive/MyDrive/NLP Assessment_3/filtered.csv', index=False)

..

In [ ]:
sentiment = pipeline(
    'text-classification',
    model='climatebert/distilroberta-base-climate-sentiment',
    return_all_scores=True
)

def get_sentiment_score(text):
    text = str(text)[:1500]
    try:
        scores = sentiment(text)[0]  # list of {label, score} dicts
        score_dict = {item['label']: item['score'] for item in scores}
        p_pos = score_dict.get('positive', 0)
        p_neg = score_dict.get('negative', 0)
        return p_pos - p_neg
    except:
        return None

print("Running ClimateBERT sentiment scoring...")
df['sentiment_score'] = df['text'].apply(get_sentiment_score)

print(f"Scored {df['sentiment_score'].notna().sum()} / {len(df)} rows successfully")
print(df['sentiment_score'].describe())

df.to_csv('/content/drive/MyDrive/NLP Assessment_3/individual_scores.csv', index=False)
print("Saved to individual_scores.csv")